In [3]:
import pandas as pd
import numpy as np

In [4]:
df_raw = pd.read_csv('../../dataset/original/renewal_calls.csv')
print(len(df_raw.columns))
print(df_raw.info())
print(df_raw.head())

C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\450473295.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../../dataset/original/renewal_calls.csv')


41
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186534 entries, 0 to 186533
Data columns (total 41 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   Call_ID                                            186534 non-null  float64
 1   Call_Direction                                     186534 non-null  object 
 2   Co_Ref                                             179149 non-null  object 
 3   Call_Date                                          186534 non-null  object 
 4   Churn_Category                                     7902 non-null    object 
 5   Complaint_Category                                 19008 non-null   object 
 6   Customer_Reaction_Category                         23085 non-null   object 
 7   Agent_Renewal_Pitch_Category                       53738 non-null   object 
 8   Customer_Renewal_Response_Category                 54312 non-null   obj

In [5]:
df_cols = df_raw.columns
print(df_cols)
# //save as a csv file
df_cols_df = pd.DataFrame(df_cols, columns=['Columns'])
df_cols_df.to_csv('../../dataset/01_raw/renewal_calls/columns.csv', index=True)

Index(['Call_ID', 'Call_Direction', 'Co_Ref', 'Call_Date', 'Churn_Category',
       'Complaint_Category', 'Customer_Reaction_Category',
       'Agent_Renewal_Pitch_Category', 'Customer_Renewal_Response_Category',
       'Agent_Response_Category', 'Membership_Renewal_Decision',
       'Serious_Complaint', 'Other_Complaint', 'Discussion_on_Price_Increase',
       'Renewal_Impact_Due_to_Price_Increase', 'Discount_or_Waiver_Requested',
       'Call_Reschedule_Request', 'Agent_Flagged_Membership_Status_Alert',
       'Agent_Renewal_Initiation', 'Explicit_Competitor_Mention',
       'Unnamed: 20', 'Explicit_Switching_Intent', 'Mentioned_Competitors',
       'Price_Switching_Mentioned', 'Competitor_Value_Comparison',
       'Competitor_Benefits_Mentioned', 'Topic_Introduced_By',
       'Percentage_Price_Increase_Mentioned',
       'Monetary_Price_Increase_Mentioned', 'Price_Range_Mentioned',
       'Customer_Asked_For_Justification', 'Customer_Response',
       'Desire_To_Cancel', 'Discount_O

## Analysing the Null Cols

In [6]:
print(df_raw.isnull().sum())

Call_ID                                                   0
Call_Direction                                            0
Co_Ref                                                 7385
Call_Date                                                 0
Churn_Category                                       178632
Complaint_Category                                   167526
Customer_Reaction_Category                           163449
Agent_Renewal_Pitch_Category                         132796
Customer_Renewal_Response_Category                   132222
Agent_Response_Category                              132554
Membership_Renewal_Decision                           99960
Serious_Complaint                                    102201
Other_Complaint                                      102202
Discussion_on_Price_Increase                          98500
Renewal_Impact_Due_to_Price_Increase                  98529
Discount_or_Waiver_Requested                          98529
Call_Reschedule_Request                 

In [7]:
null_columns = df_raw.columns[df_raw.isnull().any()]
print("Columns with null values:", null_columns)

#save the null cols as a csv file
null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns']) 
null_cols_df.to_csv('../../dataset/01_raw/renewal_calls/null_columns.csv', index=True)


Columns with null values: Index(['Co_Ref', 'Churn_Category', 'Complaint_Category',
       'Customer_Reaction_Category', 'Agent_Renewal_Pitch_Category',
       'Customer_Renewal_Response_Category', 'Agent_Response_Category',
       'Membership_Renewal_Decision', 'Serious_Complaint', 'Other_Complaint',
       'Discussion_on_Price_Increase', 'Renewal_Impact_Due_to_Price_Increase',
       'Discount_or_Waiver_Requested', 'Call_Reschedule_Request',
       'Agent_Flagged_Membership_Status_Alert', 'Agent_Renewal_Initiation',
       'Explicit_Competitor_Mention', 'Unnamed: 20',
       'Explicit_Switching_Intent', 'Mentioned_Competitors',
       'Price_Switching_Mentioned', 'Competitor_Value_Comparison',
       'Competitor_Benefits_Mentioned', 'Topic_Introduced_By',
       'Percentage_Price_Increase_Mentioned',
       'Monetary_Price_Increase_Mentioned', 'Price_Range_Mentioned',
       'Customer_Asked_For_Justification', 'Customer_Response',
       'Desire_To_Cancel', 'Discount_Offered', 'Justif

## Getting the datatype of each columns and uniques values if (more than 10 values then as ranges)

In [8]:



def is_date_column(series):
    return pd.api.types.is_datetime64_any_dtype(series)


def is_coref_column(series, threshold=50):
    num_unique = series.nunique(dropna=True)
    return series.dtype == 'object' and num_unique > threshold


def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if series.dtype == 'object':
        # Try parsing to datetime
        try:
            parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
            if all(parsed.dt.time == pd.Timestamp("00:00:00").time()):
                return "date"
            else:
                return "datetime"
        except:
            pass

        # Try parsing to time
        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return "time"
        except:
            pass

        return "categorical"

    return "categorical"


def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if is_date_column(series) or is_coref_column(series):
        return []

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f"{series.min()} - {series.max()}"

    return unique_vals.tolist()


def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "unique_values": get_unique_representation(series),
            "null_count": series.isna().sum(),
            "num_unique": series.nunique(dropna=True)
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)


def save_validation_report(df, output_path):
    dt_val_df = create_data_validation_df(df)
    print(dt_val_df)

    dt_val_df.to_csv(output_path, index=True)
    print(f"\nReport saved to: {output_path}")


# output_file = './dataset/01_raw/ad_hoc/data_validation_summary.csv'
# save_validation_report(df_raw, output_file)

In [9]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "null_count": series.isna().sum(),
            "null_percentage": round((series.isna().sum() / len(df)) * 100, 2),
            "num_unique": series.nunique(dropna=True),
            "unique_values": get_unique_representation(series),
            "top_values": series.value_counts(dropna=True).head(5).to_dict(),
            "skewness": series.skew() if pd.api.types.is_numeric_dtype(series) else None,
            "constant_column": series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)

df_validation_df = create_data_validation_df(df_raw)
print(df_validation_df.head())
df_validation_df.to_csv('../../dataset/01_raw/renewal_calls/data_validation_summary.csv', index=True)




C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\3920180566.py:20: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\3920180566.py:20:

      column_name data_type  column_type  null_count  null_percentage  \
0         Call_ID   float64    numerical           0             0.00   
1  Call_Direction    object  categorical           0             0.00   
2          Co_Ref    object  categorical        7385             3.96   
3       Call_Date    object         date           0             0.00   
4  Churn_Category    object  categorical      178632            95.76   

   num_unique                                      unique_values  \
0          53                    193000000000.0 - 695000000000.0   
1           4           [Outbound, OUT_BOUND, IN_BOUND, Inbound]   
2       36303                                                 []   
3         467                                                 []   
4          18  [Not Mentioned, Payment / Invoice Issue, Unnec...   

                                          top_values  skewness  \
0  {511000000000.0: 7183, 509000000000.0: 6542, 5... -0.734301   
1  {'Outbound': 8349

C:\Users\Asus\AppData\Local\Temp\ipykernel_34288\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')


In [10]:
def check_mixed_data_types(df):
    mixed_summary = []

    for col in df.columns:
        series = df[col].dropna()

        # Try numeric conversion
        converted = pd.to_numeric(series, errors='coerce')

        # Count valid numeric vs invalid
        numeric_count = converted.notna().sum()
        non_numeric_count = converted.isna().sum()

        # If both exist → mixed
        if numeric_count > 0 and non_numeric_count > 0:
            sample_numeric = series[converted.notna()].iloc[0]
            sample_non_numeric = series[converted.isna()].iloc[0]

            mixed_summary.append({
                "column": col,
                "numeric_count": int(numeric_count),
                "non_numeric_count": int(non_numeric_count),
                "sample_numeric": sample_numeric,
                "sample_non_numeric": sample_non_numeric
            })

    return pd.DataFrame(mixed_summary)

mixed_df = check_mixed_data_types(df_raw)
print(mixed_df)

mixed_df.to_csv('../../dataset/01_raw/renewal_calls/mixed_data_types.csv', index=False)

                        column  numeric_count  non_numeric_count  \
0  Competitor_Value_Comparison              1              88664   

  sample_numeric sample_non_numeric  
0             50     Not Applicable  


## =================================================================================